In [5]:
import pandas as pd

# List of all 12 Excel files
files = [
    "08 August 2025 Chronic Delay_0.xlsx",
    "09 September 2025 Chronic Delay_0.xlsx",
    "Table 8 Chronic Delay October 2025.xlsx",
    "Table 8 Chronic Delay November 2025.xlsx",
    "Table 8 Chronic Delay December 2025_0.xlsx",
    "Table_8_Chronic_Delay_Report_January_2026.xlsx",
    "Table_8_Chronic_Delay_Report_February_2026.xlsx",
    "Table 8 - Chronically Delay March 2026.xlsx",
    "Table 8 - April 2026 Chronically Delay.xlsx",
    "Table 8 -Chronically Delay - May 2026.xlsx",
    "Table 8 - Chronically Delay - June 2026.xlsx",
    "Table 8 - Chronically Delay - July 2026.xlsx"
]

all_data = []

for file in files:

    # Read Excel file (column headers are on row 3)
    df = pd.read_excel(file, header=2)

    # Remove empty rows and columns
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    # Older format: August–October 2025
    if "RANK" in df.columns or "Rank" in df.columns:

        df = df.iloc[:, :11].copy()

        df.columns = [
            "rank",
            "marketing_carrier_name",
            "operating_carrier_name",
            "flight_number",
            "month",
            "origin_destination",
            "departure_time",
            "total_operations",
            "delayed_operations",
            "percentage_delay",
            "average_delay_minutes"
        ]

        df["year"] = 2025
        df["marketing_carrier"] = pd.NA
        df["operating_carrier"] = pd.NA

    # Newer format: November 2025–July 2026
    else:

        df = df.iloc[:, :13].copy()

        df.columns = [
            "year",
            "marketing_carrier",
            "marketing_carrier_name",
            "operating_carrier",
            "operating_carrier_name",
            "flight_number",
            "month",
            "origin_destination",
            "departure_time",
            "total_operations",
            "delayed_operations",
            "percentage_delay",
            "average_delay_minutes"
        ]

        df["rank"] = pd.NA

    # Remove footnotes and non-flight rows
    df["flight_number"] = pd.to_numeric(
        df["flight_number"], errors="coerce"
    )

    df = df.dropna(subset=["flight_number"])

    # Convert year to numeric
    df["year"] = pd.to_numeric(
        df["year"], errors="coerce"
    )

    # Create a year-month column
    df["year_month"] = pd.to_datetime(
        df["month"].astype(str).str[:3] + " " +
        df["year"].astype(int).astype(str),
        format="%b %Y",
        errors="coerce"
    )

    # Track original file
    df["source_file"] = file

    # Add cleaned data to list
    all_data.append(df)

# Combine all 12 datasets vertically
combined = pd.concat(all_data, ignore_index=True)

# Sort by month and flight number
combined = combined.sort_values(
    ["year_month", "flight_number"]
).reset_index(drop=True)

# Export combined dataset
combined.to_csv("combined_chronic_delays.csv", index=False)

# Check results
print("Total rows:", len(combined))
print("Number of months:", combined["year_month"].nunique())

print("\nRows per month:")
print(combined.groupby("year_month").size())

print("\nFirst five rows:")
print(combined.head())


Total rows: 6752
Number of months: 12

Rows per month:
year_month
2025-08-01     469
2025-09-01     172
2025-10-01     258
2025-11-01     260
2025-12-01     990
2026-01-01     187
2026-02-01     276
2026-03-01     733
2026-04-01     319
2026-05-01     513
2026-06-01     949
2026-07-01    1626
dtype: int64

First five rows:
  rank marketing_carrier_name operating_carrier_name  flight_number month  \
0  212                 ALASKA                 ALASKA           49.0   Aug   
1  121              SOUTHWEST              SOUTHWEST          200.0   Aug   
2  231              SOUTHWEST              SOUTHWEST          203.0   Aug   
3  110              ALLEGIANT              ALLEGIANT          217.0   Aug   
4  290              SOUTHWEST              SOUTHWEST          237.0   Aug   

  origin_destination  departure_time  total_operations  delayed_operations  \
0            SEA-ANC          1055.0              19.0                11.0   
1            EUG-DEN          1455.0              11.0  